# 04 – Customer Segmentation
## FraudShield AI · K-Means Clustering + PCA

This notebook segments 2.77 million customers into behavioural clusters using
K-Means and visualises them in 2-D PCA space.

**Pipeline:**
1. Load scored transactions
2. Aggregate to customer-level profiles
3. Normalize features
4. Elbow method to determine optimal K
5. Train K-Means (K=4)
6. Map clusters to business labels
7. PCA 2-D visualisation
8. Segment profiling & business insights


In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.decomposition import PCA

from src.segmentation import (
    aggregate_customer_data, normalize_customer_features,
    run_elbow_method, train_kmeans, apply_pca, map_cluster_labels
)
from src.utils import load_model

SEGMENT_COLORS = {
    'Low-Risk Regular Customers':       '#10b981',
    'High-Value Customers':             '#3b82f6',
    'High-Value Customers (Group 2)':   '#06b6d4',
    'Fraud-Prone Customers':            '#ef4444',
    'High-Risk Customers':              '#f59e0b',
    'Dormant Customers':                '#8b5cf6',
}

print("Imports OK ✓")


## 4.1 Load Scored Transactions

In [ ]:
SCORED_PATH = "data/processed/scored_transactions.csv"
df_scored   = pd.read_csv(SCORED_PATH)
print(f"Scored dataset : {df_scored.shape[0]:,} rows")
print(f"Columns        : {list(df_scored.columns)}")
df_scored[['nameOrig','type','amount','isFraud','fraud_score','risk_category']].head(3)


## 4.2 Customer-Level Aggregation

In [ ]:
scores = df_scored['fraud_score'].values
customer_df = aggregate_customer_data(df_scored, scores)
print(f"Unique customers: {len(customer_df):,}")
print("\nCustomer profile preview:")
display(customer_df.describe().T.round(3))


## 4.3 Normalize Features

In [ ]:
SCALER_KM = "models/scaler_kmeans.pkl"
X_scaled, feat_cols = normalize_customer_features(
    customer_df, scaler_path=SCALER_KM, is_training=False
)
print(f"Scaled feature matrix: {X_scaled.shape}")
print(f"Features used: {feat_cols}")


## 4.4 Elbow Method — Optimal K

In [ ]:
ks, inertias = run_elbow_method(X_scaled, max_k=8)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, inertias, 'o-', color='#3b82f6', linewidth=2.5, markersize=8,
        markerfacecolor='#06b6d4', markeredgecolor='white', markeredgewidth=1.5)
ax.axvline(4, color='#ef4444', linestyle='--', linewidth=1.5, alpha=0.8,
            label='K=4 (selected)')
ax.fill_between(ks, inertias, alpha=0.1, color='#3b82f6')
ax.set_xlabel('Number of Clusters (K)', fontsize=11)
ax.set_ylabel('Inertia (WCSSE)', fontsize=11)
ax.set_title('Elbow Method for Optimal K', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('reports/figures/nb_elbow.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nElbow clearly bends at K=4 → selected as optimal")


## 4.5 Train K-Means (K=4)

In [ ]:
kmeans = load_model("models/kmeans.pkl")   # use the pre-trained model
print(f"K-Means loaded: {kmeans.n_clusters} clusters")

customer_df['cluster'] = kmeans.predict(X_scaled)
print("\nCluster counts:")
print(customer_df['cluster'].value_counts().sort_index())


## 4.6 Business Label Mapping

In [ ]:
customer_df, label_mapping = map_cluster_labels(customer_df, kmeans, feat_cols)
print("\nCluster → Business Label mapping:")
for k, v in label_mapping.items():
    print(f"  Cluster {k}  →  {v}")

print("\nSegment distribution:")
display(customer_df['segment_label'].value_counts().to_frame('Count'))


## 4.7 PCA 2-D Visualisation

In [ ]:
X_pca, pca = apply_pca(X_scaled)
customer_df['pca1'] = X_pca[:, 0]
customer_df['pca2'] = X_pca[:, 1]

var1, var2 = pca.explained_variance_ratio_ * 100
print(f"PC1 variance: {var1:.1f}%  |  PC2 variance: {var2:.1f}%  |  Total: {var1+var2:.1f}%")

# Downsample for plotting
PLOT_N  = min(30_000, len(customer_df))
plot_df = customer_df.sample(n=PLOT_N, random_state=42)

fig, ax = plt.subplots(figsize=(11, 8))
for seg in sorted(plot_df['segment_label'].unique()):
    mask = plot_df['segment_label'] == seg
    color = SEGMENT_COLORS.get(seg, '#94a3b8')
    ax.scatter(plot_df.loc[mask, 'pca1'], plot_df.loc[mask, 'pca2'],
               c=color, s=3, alpha=0.55, label=seg, edgecolors='none')

ax.set_xlabel(f'PC1 ({var1:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({var2:.1f}% variance)', fontsize=11)
ax.set_title(f'Customer Segments in PCA 2-D Space  (n={PLOT_N:,} sampled)', fontsize=13)
ax.legend(markerscale=5, fontsize=9, loc='upper right',
           framealpha=0.3, facecolor='#0a0e1a', edgecolor='#334155')
plt.tight_layout()
plt.savefig('reports/figures/nb_pca_clusters.png', dpi=150, bbox_inches='tight')
plt.show()


## 4.8 Segment Profile Analysis

In [ ]:
profile_cols = ['transaction_frequency', 'average_transaction_amount',
               'total_spending', 'fraud_probability_score',
               'number_of_suspicious_transactions', 'fraud_count', 'average_balance']

profile = customer_df.groupby('segment_label')[profile_cols].mean().round(3)
display(profile)


In [ ]:
# Radar chart
from matplotlib.patches import FancyArrowPatch
angles     = np.linspace(0, 2*np.pi, len(profile_cols), endpoint=False).tolist()
radar_cols = ['Tx Freq', 'Avg Amt', 'Total Spend', 'Fraud Score',
              'Suspicious Txs', 'Fraud Count', 'Avg Balance']

# Normalize 0-1
norm_profile = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for seg in norm_profile.index:
    vals = norm_profile.loc[seg].tolist() + [norm_profile.loc[seg].tolist()[0]]
    _angles = angles + [angles[0]]
    color = SEGMENT_COLORS.get(seg, '#94a3b8')
    ax.plot(_angles, vals, color=color, linewidth=2, label=seg)
    ax.fill(_angles, vals, color=color, alpha=0.12)

ax.set_thetagrids(np.degrees(angles), radar_cols, fontsize=9)
ax.set_title('Segment Profiles (Normalised)', fontsize=13, y=1.08)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1),
           fontsize=8, framealpha=0.3, facecolor='#111827', edgecolor='#334155')
plt.tight_layout()
plt.savefig('reports/figures/nb_segment_radar.png', dpi=150, bbox_inches='tight')
plt.show()


## 4.9 Business Segment Insights

| Segment | Key Characteristics | Business Action |
|---|---|---|
| **Low-Risk Regular Customers** | Low tx frequency, small amounts, zero fraud history | Standard service; loyalty rewards |
| **High-Value Customers** | Large balances, high total spending, rare fraud | Premium customer care; escalated alerts |
| **High-Value Customers (Group 2)** | Multi-transaction, high-volume | VIP tier monitoring |
| **Fraud-Prone Customers** | High fraud scores, confirmed fraud history | Flag for investigation; transaction limits |

> **K-Means K=4** chosen by elbow method.
> **PCA** retains 71.5% of variance in 2 dimensions.


In [ ]:
# Save the enriched customer dataframe (optional — already saved by pipeline)
SAVE_PATH = "data/processed/customer_segments_notebook.csv"
customer_df.to_csv(SAVE_PATH, index=False)
print(f"Saved enriched segments to {SAVE_PATH}")
print("\nNotebook 04 complete ✓")
